# Ham and Spam email classification
## EDA
- First we perform EDA with the input data
- Following actions are performed:
    - Check for missing values
    - Distribution of target labels
    - Histogram of wordcounts
    - Most common words
    - Domain analysis

In [1]:
import numpy as np
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re
from sklearn.feature_extraction.text import CountVectorizer
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score


In [3]:
df = pd.read_csv("emails.csv")
df.head()


,Unnamed: 0,filename,contents,target
0,0,00249.5f45607c1bffe89f60ba1ec9f878039a,"Dear Homeowner,\n \nInterest Rates are at thei...",1
1,1,0355.94ebf637e4bd3db8a81c8ce68ecf681d,"Friend,Now you can copy DVD's and Games\nhttp:...",1
2,2,0395.bb934e8b4c39d5eab38f828a26f760b4,Pocket the newest 8 year annuity!\t Pocket th...,1
3,3,0485.9021367278833179285091e5201f5854,Congratulations! You Get a Free Handheld Organ...,1
4,4,00373.ebe8670ac56b04125c25100a36ab0510,ATTENTION: This is a MUST for ALL Computer Use...,1


In [4]:
# 1. Basic dataset overview
print(f"Dataset dimensions: {df.shape[0]} emails, {df.shape[1]} features")

Dataset dimensions: 9353 emails, 4 features


- There are 9353 emails
- Althought there are 4 features, the input is the email content which is the **contents** columns and the output is the **target** columns
- The target column contains only binary value with 0: Ham and 1: Spam email

In [5]:
# 2. Missing values analysis
missing_values = np.array([not isinstance(text, str) for text in df['contents'].values])
missing_count = np.sum(missing_values)
print(f"Missing values in content: {missing_count} ({missing_count/len(df):.2%})")

Missing values in content: 146 (1.56%)


In [8]:
146/9353

0.015609964717203037

In [7]:
df[missing_values]

,Unnamed: 0,filename,contents,target
13,13,0146.6656452972931e859e640f6ac57d2962,NaN,1
78,78,00139.b2a205ac25d7d907cdfb3f865dbae1ae,NaN,1
169,169,0352.f7adb4aa267e50a8db1e4bcacfe863f3,NaN,1
192,192,00341.99b463b92346291f5848137f4a253966,NaN,1
282,282,0340.8e191c37e2d30a639013203aacf60086,NaN,1
...,...,...,...,...
9057,9057,00565.630d62a91f6d1b297a2069007700e2ae,NaN,0
9126,9126,00653.41f993b29996e0e099849f377ae280e4,NaN,0
9206,9206,01285.d0d97662f93c959244325cc414f96039,NaN,0
9306,9306,01306.642dc8dce3771af294de84e167221354,NaN,0


In [ ]:
df.target.value_counts(normalize=True).plot(kind='bar', color=['#1f77b4', '#ff7f0e'])
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()


- We can see that more than 70% are No Spam email and less than 30% are classified as Spam email
- Next we will see the distribution of number of words

In [36]:
df['word_count'] = df['contents'].apply(lambda x: len(str(x).split()))


In [ ]:
df.head()

In [ ]:
# Plot histograms
plt.figure(figsize=(10, 6))
plt.hist(df[df['target'] == 0]['word_count'], bins=50, alpha=0.6, label='Target = Ham',color="green")
plt.hist(df[df['target'] == 1]['word_count'], bins=50, alpha=0.6, label='Target = Spam',color="red")
plt.title('Histogram of Word Count by Target Label')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True)
plt.show()

- Both Ham and Spam document have extreme right skew
- More Ham emails than Spam 
- Ham emails tend to be longer than Spam
- Most of the documents have less than 12000 words, but there still exits documents with lots of words
- Ham documents have more words than Spam for most email text length

Next we will compute the most common words in n-grams:

In [46]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # Convert to lowercase and remove non-alphanumeric
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    # Remove stopwords
    words = [word for word in text.split() if word not in stop_words and len(word) > 2]
    return ' '.join(words)

# Preprocess text
processed_ham = [preprocess_text(text) for text, label in zip(df['contents'].values, labels) if label == 0]
processed_spam = [preprocess_text(text) for text, label in zip(df['contents'].values, labels) if label == 1]

# Extract most common words
vectorizer = CountVectorizer(max_features=20)
ham_counts = vectorizer.fit_transform(processed_ham)
ham_words = vectorizer.get_feature_names_out()
ham_freq = np.asarray(ham_counts.sum(axis=0)).ravel()

vectorizer = CountVectorizer(max_features=20)
spam_counts = vectorizer.fit_transform(processed_spam)
spam_words = vectorizer.get_feature_names_out()
spam_freq = np.asarray(spam_counts.sum(axis=0)).ravel()


In [ ]:

# Plot most common words
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Ham words
ax1.barh(ham_words, ham_freq, color='green')
ax1.set_title('Most Common Words in Ham Emails')
ax1.set_xlabel('Frequency')

# Spam words
ax2.barh(spam_words, spam_freq, color='red')
ax2.set_title('Most Common Words in Spam Emails')
ax2.set_xlabel('Frequency')

plt.tight_layout()
plt.show()

Words Cloud:

In [ ]:
plt.figure(figsize=(16, 7))

# Ham wordcloud
ham_text = ' '.join(processed_ham)
ham_wordcloud = WordCloud(width=800, height=400, background_color='white', 
                         max_words=100, contour_width=3, contour_color='steelblue')
ham_wordcloud.generate(ham_text)

plt.subplot(1, 2, 1)
plt.imshow(ham_wordcloud, interpolation='bilinear')
plt.title('Ham Email Word Cloud')
plt.axis('off')

# Spam wordcloud
spam_text = ' '.join(processed_spam)
spam_wordcloud = WordCloud(width=800, height=400, background_color='white',
                          max_words=100, contour_width=3, contour_color='darkred')
spam_wordcloud.generate(spam_text)

plt.subplot(1, 2, 2)
plt.imshow(spam_wordcloud, interpolation='bilinear')
plt.title('Spam Email Word Cloud')
plt.axis('off')

plt.tight_layout()
plt.show()


- Missing values

Domain data analysis

In [ ]:

# 9. Domain analysis (checking sender domains if available)
email_pattern = re.compile(r'[\w\.-]+@[\w\.-]+')
domains = []
for content in df['contents'].values:
    if isinstance(content, str):
        emails = email_pattern.findall(content)
        domains.extend([email.split('@')[1] for email in emails if '@' in email])

top_domains = Counter(domains).most_common(15)
if top_domains:
    domain_names, domain_counts = zip(*top_domains)
    
    plt.figure(figsize=(12, 6))
    plt.barh(domain_names, domain_counts)
    plt.xlabel('Frequency')
    plt.ylabel('Domain')
    plt.title('Most Common Email Domains')
    plt.tight_layout()
    plt.show()

### Modeling

In [14]:
# Fill NaN values with empty strings
df['contents'] = df['contents'].fillna('')

X = df.contents
y = df.target

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=123)



In [ ]:
X_train.head()

In [ ]:

# Create text classification pipeline
text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', MultinomialNB(alpha=0.1))
])


# Apply K-Fold Cross-validation on training data
k_fold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(text_clf, X_train, y_train, cv=k_fold, scoring='accuracy')

# Fit final model on entire training set
text_clf.fit(X_train, y_train)

In [ ]:
# Evaluate on test set
y_pred = text_clf.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

# Output results
print(f"Cross-validation accuracy: {np.mean(cv_scores):.4f} (±{np.std(cv_scores):.4f})")
print(f"Test accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:

# Feature importance analysis (log-probability ratios)
feature_names = text_clf.named_steps['tfidf'].get_feature_names_out()
log_probs = text_clf.named_steps['clf'].feature_log_prob_
spam_features = log_probs[1] - log_probs[0]
indices = np.argsort(spam_features)[-20:]  # Top 20 spam-indicative features
top_spam_features = [(feature_names[i], spam_features[i]) for i in indices]
print("\nTop spam-indicative features:")
for feature, score in top_spam_features:
    print(f"{feature}: {score:.4f}")